# dispatch-back-fn-from-recipe — ex3: wildcard (fn, None) fallback when (fn, argnum) is unregistered

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `dispatch-back-fn-from-recipe`. Running the final beacon cell reports progress against the `Backprop: dispatch back fn from recipe` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: dispatch back fn from recipe` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dispatch-back-fn-from-recipe`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dispatch-back-fn-from-recipe"
DD_SUBTOPIC = "Backprop: dispatch back fn from recipe"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Wildcard fallback — `(fn, None)` matches any argnum

Ex1 looked up `(recipe.func, argnum)`; ex2 wrapped the lookup with a
friendly KeyError. The deepening move handles a real registry pattern:
some back_fns are SYMMETRIC across all argnums (e.g. `add_back` for
`x + y` returns the same `grad_out` regardless of argnum). The
registry stores them as `(fn, None)` and the dispatcher falls back to
this wildcard when no exact match is found.

```python
back_funcs = {
    (multiply, 0): mul_back_0,   # specific — needs y
    (multiply, 1): mul_back_1,   # specific — needs x
    (add, None):   add_back,     # WILDCARD — same for any argnum
}

def lookup(fn, argnum):
    if (fn, argnum) in back_funcs:
        return back_funcs[(fn, argnum)]   # exact match wins
    if (fn, None) in back_funcs:
        return back_funcs[(fn, None)]     # wildcard fallback
    raise KeyError(f'No back_fn for ({fn.__name__}, {argnum})')
```

**Precedence: exact before wildcard.** If both `(add, 0)` and
`(add, None)` are registered, `argnum=0` MUST resolve to the exact
match. The wildcard only fires when no specific entry exists.

**Why this is the production pattern.** Hand-registering `(fn, 0)`,
`(fn, 1)`, ..., `(fn, k)` for every k-ary symmetric op is bloat. The
wildcard lets the registry stay sparse: one entry per op family, with
argnum-specific overrides only where the math differs.

### Exercise 3 — wildcard (fn, None) fallback when (fn, argnum) is unregistered

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply a two-step lookup that first probes `(recipe.func, argnum)` for an exact back_fn, then falls back to `(recipe.func, None)` as a wildcard — enforcing exact-match precedence over wildcard, raising `KeyError` only when neither key is registered.
> Keywords: dispatch, wildcard, fallback, back-funcs, precedence
> ```

**KCs targeted:** `dispatch-back-fn-from-recipe`, `exact-before-wildcard-precedence`

Implement `ex3_dispatch_with_wildcard(node, back_funcs)` — same return shape as ex1's `dispatch_back_fns` (`list[(argnum, parent, back_fn)]`), but with a two-step lookup per parent:

For each `(argnum, parent)` in `node.recipe.parents`:

1. **Exact match.** If `(node.recipe.func, argnum) in back_funcs`, use that back_fn.
2. **Wildcard fallback.** Otherwise, if `(node.recipe.func, None) in back_funcs`, use that back_fn.
3. **Neither.** Raise `KeyError(f'No back_fn for ({fn_name}, {argnum})')` where `fn_name = getattr(node.recipe.func, '__name__', repr(node.recipe.func))`.

**Precedence is mandatory.** If both `(fn, 0)` and `(fn, None)` exist, `argnum=0` MUST resolve to the EXACT match. The wildcard is a fallback, not an override.

**Use case.** Symmetric ops like `add(x, y)` have the SAME back_fn for argnum=0 and argnum=1: `add_back(grad_out, out) = grad_out` regardless of which parent. Registering it once as `(add, None)` instead of duplicating for every argnum keeps the registry sparse.

Assume `node.recipe is not None` (the caller filters leaves).

In [ ]:
def ex3_dispatch_with_wildcard(node, back_funcs):
    results = []
    fn = node.recipe.func
    for argnum, parent in node.recipe.parents.items():
        if (fn, argnum) in back_funcs:
            back_fn = back_funcs[(fn, argnum)]            # exact wins
        elif (fn, None) in back_funcs:
            back_fn = back_funcs[(fn, None)]              # wildcard fallback
        else:
            fn_name = getattr(fn, '__name__', repr(fn))
            raise KeyError(
                f'No back_fn for ({fn_name}, {argnum})'
            )
        results.append((argnum, parent, back_fn))
    return results


<details><summary>Solution</summary>

```python
def ex3_dispatch_with_wildcard(node, back_funcs):
    results = []
    fn = node.recipe.func
    for argnum, parent in node.recipe.parents.items():
        if (fn, argnum) in back_funcs:
            back_fn = back_funcs[(fn, argnum)]            # exact wins
        elif (fn, None) in back_funcs:
            back_fn = back_funcs[(fn, None)]              # wildcard fallback
        else:
            fn_name = getattr(fn, '__name__', repr(fn))
            raise KeyError(
                f'No back_fn for ({fn_name}, {argnum})'
            )
        results.append((argnum, parent, back_fn))
    return results
```

**`in back_funcs` is the right probe.** `back_funcs.get((fn, argnum))` and checking for `None` would conflate 'unregistered' with 'registered as None'. Membership check is unambiguous.

**Order matters: exact before wildcard.** An `if/elif` chain enforces this naturally — once `(fn, argnum)` is found, the wildcard branch never executes. Reversing the order would make every wildcard registration shadow every exact one — a bug.

**Why `getattr(fn, '__name__', repr(fn))` in the error.** Some registry keys are strings (e.g. `'add'` in our tests), not callables. `repr('add')` yields `"'add'"`. `getattr` returns the string directly when no `__name__` attribute exists (well, strings DO have `__name__`? — actually no, strings don't have `__name__`, so this falls through to `repr(fn)`). For real callables, `fn.__name__` is what users recognize.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()